In [1]:
# set up environment
import sys
import json
import re
import requests
import pandas as pd
from pathlib import Path
import json
import csv
from caveclient import CAVEclient

import urllib.parse

from pathlib import Path
#pip install neuprint-python
from neuprint import Client, fetch_neurons, fetch_custom, NeuronCriteria as NC


print("Python executable:", sys.executable)
print("Imports OK")

c:\Users\jsfdz\Documents\Python\cave_env\lib\site-packages\neuprint\utils.py:89: UserWarning: 
Progress bar will not work well in the notebook without ipywidgets.
Run the following commands (for notebook and jupyterlab users):

    conda install -c conda-forge ipywidgets
    jupyter nbextension enable --py widgetsnbextension
    jupyter labextension install @jupyter-widgets/jupyterlab-manager

...and then reload your jupyter session, and restart your kernel.

  warnings.warn(msg)


Python executable: c:\Users\jsfdz\Documents\Python\cave_env\Scripts\python.exe
Imports OK


In [2]:
# set up directories
PROJECT_ROOT = Path.cwd() #anchor to current notebook location


OUTPUT_DIR = PROJECT_ROOT / "output-MANCv1.2.3_04B_add neuromeres T2R 20260413"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


print("Output:", OUTPUT_DIR)



Output: c:\Users\jsfdz\4B-analysis\4B-analysis\output-MANCv1.2.3_04B_add neuromeres T2R 20260413


In [3]:
from dotenv import load_dotenv
import os

load_dotenv()  # loads .env from current working directory

token = os.environ["NEUPRINT_APPLICATION_CREDENTIALS"]

In [4]:
from neuprint import Client

c = Client(
    "https://neuprint.janelia.org",
    dataset="manc:v1.2.3",
    token=token
)
print(c)

Client("https://neuprint.janelia.org", "manc:v1.2.3")


In [5]:
#Fetch neurons to populate the state later: one neuromere with 04B add T2 right side
from neuprint import fetch_custom

cypher = """
MATCH (n:Neuron)
WHERE n.hemilineage = '04B'
  AND n.somaNeuromere = 'T2'
  AND n.somaSide = 'RHS'
RETURN
  n.bodyId AS bodyId,
  n.type AS type,
  n.instance AS instance,
  n.class AS class,
  n.subclass AS subclass,
  n.hemilineage AS hemilineage,
  n.somaNeuromere AS somaNeuromere,
  n.somaSide AS somaSide,
  n.rootSide AS rootSide,
  n.longTract AS longTract,
  n.birthtime AS birthtime
ORDER BY bodyId
"""

df_4b_t2_rhs = fetch_custom(cypher, client=c)
df_4b_t2_rhs.head() 
df_4b_t2_rhs["bodyId"].nunique() # n=97

95

In [25]:
# Generate a new state that diaplays the 4B morphology clusters, as layers for each subclass WITH  all neurons in one layer displayed with the same color. 04BT2l

import json
import urllib.parse
from itertools import cycle


#generate new, clean state:

##define sources:
# 1) Paste a known-good MANC Clio URL here (we wont repopulate it; it is jsut for retrieving sources)
BASE_URL = "https://clio-ng.janelia.org/#!%7B%22title%22:%22manc-v1.2.3-neuprint-layers%22%2C%22dimensions%22:%7B%22x%22:%5B8e-9%2C%22m%22%5D%2C%22y%22:%5B8e-9%2C%22m%22%5D%2C%22z%22:%5B8e-9%2C%22m%22%5D%7D%2C%22position%22:%5B23056.5%2C29733.5%2C41138.5%5D%2C%22crossSectionOrientation%22:%5B0%2C0.7071067690849304%2C-0.7071067690849304%2C0%5D%2C%22crossSectionScale%22:1%2C%22projectionOrientation%22:%5B0%2C0.7071067690849304%2C-0.7071067690849304%2C0%5D%2C%22projectionScale%22:91364.04452716278%2C%22layers%22:%5B%7B%22type%22:%22image%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://flyem-vnc-2-26-213dba213ef26e094c16c860ae7f4be0/v3_emdata_clahe_xy/jpeg%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22tab%22:%22rendering%22%2C%22name%22:%22em%22%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%5B%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22subsources%22:%7B%22default%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%22%2C%22enableDefaultSubsources%22:false%7D%5D%2C%22toolBindings%22:%7B%22Q%22:%22selectSegments%22%7D%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%2210007%22%5D%2C%22segmentColors%22:%7B%2233946%22:%22#808080%22%7D%2C%22name%22:%22manc:v1.2.3%22%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://flyem-vnc-roi-d5f392696f7a48e27f49fa1a9db5ee3b/all-vnc-roi%22%2C%22subsources%22:%7B%22default%22:true%2C%22properties%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22pick%22:false%2C%22tab%22:%22rendering%22%2C%22selectedAlpha%22:0%2C%22saturation%22:0%2C%22meshSilhouetteRendering%22:4%2C%22segments%22:%5B%221%22%5D%2C%22segmentDefaultColor%22:%22#ffffff%22%2C%22name%22:%22all-tissue%22%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://flyem-vnc-roi-d5f392696f7a48e27f49fa1a9db5ee3b/synaptic-neuropil%22%2C%22subsources%22:%7B%22default%22:true%2C%22properties%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22pick%22:false%2C%22tab%22:%22rendering%22%2C%22selectedAlpha%22:0%2C%22saturation%22:0%2C%22meshSilhouetteRendering%22:4%2C%22segments%22:%5B%221%22%5D%2C%22segmentDefaultColor%22:%22#ffffff%22%2C%22segmentColors%22:%7B%221%22:%22#ffffff%22%7D%2C%22name%22:%22all-synaptic%22%2C%22archived%22:true%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://flyem-vnc-roi-d5f392696f7a48e27f49fa1a9db5ee3b/roi-202208%22%2C%22subsources%22:%7B%22default%22:true%2C%22properties%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22pick%22:false%2C%22tab%22:%22segments%22%2C%22selectedAlpha%22:0%2C%22saturation%22:0.5%2C%22meshSilhouetteRendering%22:4%2C%22segments%22:%5B%2210%22%2C%2211%22%2C%2212%22%2C%2213%22%2C%2214%22%2C%2215%22%2C%2216%22%2C%2217%22%2C%2218%22%2C%2219%22%2C%2220%22%2C%2221%22%2C%2222%22%2C%2223%22%2C%2224%22%2C%2225%22%2C%2226%22%2C%2227%22%2C%224%22%2C%225%22%2C%226%22%2C%227%22%2C%228%22%2C%229%22%5D%2C%22name%22:%22neuropils%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://flyem-vnc-roi-d5f392696f7a48e27f49fa1a9db5ee3b/nerve-roi-202301%22%2C%22subsources%22:%7B%22default%22:true%2C%22properties%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22pick%22:false%2C%22tab%22:%22segments%22%2C%22objectAlpha%22:0.1%2C%22segments%22:%5B%221%22%2C%2210%22%2C%2211%22%2C%2212%22%2C%2213%22%2C%2214%22%2C%2215%22%2C%2216%22%2C%2217%22%2C%2218%22%2C%2219%22%2C%222%22%2C%2220%22%2C%2221%22%2C%2222%22%2C%2223%22%2C%2224%22%2C%2225%22%2C%2226%22%2C%2227%22%2C%2228%22%2C%2229%22%2C%223%22%2C%2230%22%2C%2231%22%2C%2232%22%2C%2233%22%2C%2234%22%2C%2235%22%2C%224%22%2C%225%22%2C%226%22%2C%227%22%2C%228%22%2C%229%22%5D%2C%22name%22:%22nerves%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22annotation%22%2C%22source%22:%22precomputed://gs://manc-seg-v1p2/manc-v1.2-synapse-partners-minconf-0.0.precomputed%22%2C%22tab%22:%22rendering%22%2C%22ignoreNullSegmentFilter%22:false%2C%22shader%22:%22#uicontrol%20bool%20showPre%20checkbox%28default=true%29%5Cn#uicontrol%20bool%20showPostPartners%20checkbox%28default=true%29%5Cn%5Cn#uicontrol%20vec3%20preColor%20color%28default=%5C%22red%5C%22%29%5Cn#uicontrol%20vec3%20postColor%20color%28default=%5C%22blue%5C%22%29%5Cn%5Cn//%20Note:%5Cn//%20%20%20For%20the%20MANC%20in%20neuprint%2C%5Cn//%20%20%20connection%20strengths%20don%27t%5Cn//%20%20%20include%20any%20synapses%20below%5Cn//%20%20%20confidence%200.4.%5Cn#uicontrol%20float%20preConfidence%20slider%28min=0%2C%20max=1%2C%20default=0%29%5Cn#uicontrol%20float%20postConfidence%20slider%28min=0%2C%20max=1%2C%20default=0%29%5Cn%5Cnvoid%20main%28%29%20%7B%5Cn%20%20setColor%28defaultColor%28%29%29%3B%5Cn%5Cn%20%20float%20preSize%20=%206.0%3B%5Cn%20%20float%20postSize%20=%204.0%3B%5Cn%20%20%5Cn%20%20setEndpointMarkerColor%28%5Cn%20%20%20%20vec4%28preColor%2C%200.5%29%2C%5Cn%20%20%20%20vec4%28postColor%2C%200.5%29%29%3B%5Cn%20%20%5Cn%20%20if%20%28showPre%20&&%20showPostPartners%29%20%7B%5Cn%20%20%20%20setLineWidth%281.0%29%3B%5Cn%20%20%20%20setEndpointMarkerSize%28preSize%2C%20postSize%29%3B%5Cn%20%20%7D%5Cn%20%20else%20if%20%28showPostPartners%29%20%7B%5Cn%20%20%20%20setLineWidth%280.0%29%3B%5Cn%20%20%20%20setEndpointMarkerSize%280.0%2C%20postSize%29%3B%5Cn%20%20%7D%5Cn%20%20else%20if%20%28showPre%29%20%7B%5Cn%20%20%20%20setLineWidth%280.0%29%3B%5Cn%20%20%20%20setEndpointMarkerSize%28preSize%2C%200.0%29%3B%5Cn%20%20%7D%5Cn%20%20else%20%7B%5Cn%20%20%20%20setLineWidth%280.0%29%3B%5Cn%20%20%20%20setEndpointMarkerSize%280.0%2C%200.0%29%3B%5Cn%20%20%7D%5Cn%20%20%5Cn%20%20if%20%28prop_pre_synaptic_confidence%28%29%20%3C%20preConfidence%20%7C%7C%5Cn%20%20%20%20%20%20prop_post_synaptic_confidence%28%29%20%3C%20postConfidence%29%20discard%3B%5Cn%7D%5Cn%22%2C%22shaderControls%22:%7B%22showPostPartners%22:false%2C%22postColor%22:%22#00ffff%22%2C%22preConfidence%22:0.4%2C%22postConfidence%22:0.4%7D%2C%22linkedSegmentationLayer%22:%7B%22pre_synaptic_cell%22:%22manc:v1.2.3%22%7D%2C%22filterBySegmentation%22:%5B%22pre_synaptic_cell%22%5D%2C%22name%22:%22presyn%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22annotation%22%2C%22source%22:%22precomputed://gs://manc-seg-v1p2/manc-v1.2-synapse-partners-minconf-0.0.precomputed%22%2C%22tab%22:%22rendering%22%2C%22ignoreNullSegmentFilter%22:false%2C%22shader%22:%22#uicontrol%20bool%20showPrePartners%20checkbox%28default=true%29%5Cn#uicontrol%20bool%20showPost%20checkbox%28default=true%29%5Cn%5Cn#uicontrol%20vec3%20preColor%20color%28default=%5C%22red%5C%22%29%5Cn#uicontrol%20vec3%20postColor%20color%28default=%5C%22blue%5C%22%29%5Cn%5Cn//%20Note:%5Cn//%20%20%20For%20the%20MANC%20in%20neuprint%2C%5Cn//%20%20%20connection%20strengths%20don%27t%5Cn//%20%20%20include%20any%20synapses%20below%5Cn//%20%20%20confidence%200.4.%5Cn#uicontrol%20float%20preConfidence%20slider%28min=0%2C%20max=1%2C%20default=0.4%29%5Cn#uicontrol%20float%20postConfidence%20slider%28min=0%2C%20max=1%2C%20default=0.4%29%5Cn%5Cnvoid%20main%28%29%20%7B%5Cn%20%20setColor%28defaultColor%28%29%29%3B%5Cn%5Cn%20%20float%20preSize%20=%206.0%3B%5Cn%20%20float%20postSize%20=%204.0%3B%5Cn%20%20%5Cn%20%20setEndpointMarkerColor%28%5Cn%20%20%20%20vec4%28preColor%2C%200.5%29%2C%5Cn%20%20%20%20vec4%28postColor%2C%200.5%29%29%3B%5Cn%20%20%5Cn%20%20if%20%28showPrePartners%20&&%20showPost%29%20%7B%5Cn%20%20%20%20setLineWidth%281.0%29%3B%5Cn%20%20%20%20setEndpointMarkerSize%28preSize%2C%20postSize%29%3B%5Cn%20%20%7D%5Cn%20%20else%20if%20%28showPost%29%20%7B%5Cn%20%20%20%20setLineWidth%280.0%29%3B%5Cn%20%20%20%20setEndpointMarkerSize%280.0%2C%20postSize%29%3B%5Cn%20%20%7D%5Cn%20%20else%20if%20%28showPrePartners%29%20%7B%5Cn%20%20%20%20setLineWidth%280.0%29%3B%5Cn%20%20%20%20setEndpointMarkerSize%28preSize%2C%200.0%29%3B%5Cn%20%20%7D%5Cn%20%20else%20%7B%5Cn%20%20%20%20setLineWidth%280.0%29%3B%5Cn%20%20%20%20setEndpointMarkerSize%280.0%2C%200.0%29%3B%5Cn%20%20%7D%5Cn%20%20%5Cn%20%20if%20%28prop_pre_synaptic_confidence%28%29%20%3C%20preConfidence%20%7C%7C%5Cn%20%20%20%20%20%20prop_post_synaptic_confidence%28%29%20%3C%20postConfidence%29%20discard%3B%5Cn%7D%5Cn%22%2C%22shaderControls%22:%7B%22showPrePartners%22:false%2C%22postColor%22:%22#00ffff%22%7D%2C%22linkedSegmentationLayer%22:%7B%22post_synaptic_cell%22:%22manc:v1.2.3%22%7D%2C%22filterBySegmentation%22:%5B%22post_synaptic_cell%22%5D%2C%22name%22:%22postsyn%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://vnc-v3-seg-3d2f1c08fd4720848061f77362dc6c17/mask%22%2C%22subsources%22:%7B%22default%22:true%2C%22properties%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22pick%22:false%2C%22tab%22:%22segments%22%2C%22segmentColors%22:%7B%221%22:%22#000000%22%7D%2C%22name%22:%22voxel-classes%22%2C%22archived%22:true%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://vnc-v3-seg-3d2f1c08fd4720848061f77362dc6c17/rc5_wsexp%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22tab%22:%22segments%22%2C%22name%22:%22initial-supervoxels%22%2C%22archived%22:true%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://flyem-vnc-roi-d5f392696f7a48e27f49fa1a9db5ee3b/court-et-al-systematic-manc_tracts/mesh%22%2C%22subsources%22:%7B%22default%22:true%2C%22properties%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22tab%22:%22rendering%22%2C%22saturation%22:0.5%2C%22meshSilhouetteRendering%22:4%2C%22segments%22:%5B%22100%22%2C%22101%22%2C%22102%22%2C%22103%22%2C%22104%22%2C%22105%22%2C%22106%22%5D%2C%22name%22:%22court%20et%20al.%20tracts%22%2C%22archived%22:true%7D%5D%2C%22showSlices%22:false%2C%22prefetch%22:false%2C%22selectedLayer%22:%7B%22flex%22:1.49%2C%22size%22:426%2C%22visible%22:true%2C%22layer%22:%22manc:v1.2.3%22%7D%2C%22projectionBackgroundColor%22:%22#ffffff%22%2C%22layout%22:%223d%22%2C%22settingsPanel%22:%7B%22visible%22:true%7D%2C%22selection%22:%7B%22flex%22:0.51%2C%22size%22:426%2C%22visible%22:false%7D%7D"

# 2) Decode its state
encoded_state = BASE_URL.split("#!", 1)[1]
base_state = json.loads(urllib.parse.unquote(encoded_state))

# 3) Extract the actual working sources from that scene
em_source = None
seg_source = None

for L in base_state["layers"]:
    if L.get("type") == "image" and em_source is None:
        em_source = L.get("source")
    if L.get("type") == "segmentation" and seg_source is None:
        seg_source = L.get("source")

print("EM source:", em_source)
print("Seg source:", seg_source)

##generate state
new_state = {
    "title": "MANC_v1.2.3_4B_T2r",
    "dimensions": base_state["dimensions"],
    "position": base_state["position"],
    "crossSectionOrientation": base_state["crossSectionOrientation"],
    "crossSectionScale": base_state["crossSectionScale"],
    "projectionOrientation": base_state["projectionOrientation"],
    "projectionScale": base_state["projectionScale"],
    "layout": base_state.get("layout", "3d"),
    "showSlices": base_state.get("showSlices", True),
    "projectionBackgroundColor": base_state.get("projectionBackgroundColor", "#ffffff"),
    "crossSectionBackgroundColor": base_state.get("crossSectionBackgroundColor", "#ffffff"),
    "selectedLayer": base_state.get("selectedLayer"),
    "settingsPanel": base_state.get("settingsPanel", {"visible": True}),
    "layers": [
        {
            "type": "image",
            "name": "EM",
            "source": em_source,
            "visible": True
        }
    ]
}


# --- Color palette (cycles if needed) ---
COLORS = [
    "#87CEEB",  # sky blue
    "#FFA500",  # orange
    "#E34234",  # vermillion
    "#CC99FF",  # pale violet
    "#66CDAA",  # medium aquamarine
    "#FFD700",  # gold
]

color_cycle = cycle(COLORS)


# --- Helper: upsert layer (FIXED) ---
def add_segmentation_layer(state, name, body_ids, seg_source, color):
    body_ids = sorted(set(int(x) for x in body_ids))

    layer = {
        "type": "segmentation",
        "name": name,
        "source": seg_source,
        "segments": [str(x) for x in body_ids],
        "segmentColors": {str(x): color for x in body_ids},
        "visible": True,
    }

    # replace if exists
    for i, existing in enumerate(state["layers"]):
        if existing.get("name") == name:
            state["layers"][i] = layer
            return

    # otherwise append
    state["layers"].append(layer)


# --- Clean subclass labels ---
subclass_df = df_4b_t2_rhs.copy()
subclass_df["subclass"] = (
    subclass_df["subclass"]
    .fillna("unclassified")
    .astype(str)
    .str.strip()
)
subclass_df.loc[subclass_df["subclass"] == "", "subclass"] = "unclassified"


# --- Inspect counts ---
subclass_counts = (
    subclass_df.groupby("subclass")["bodyId"]
    .nunique()
    .sort_values(ascending=False)
)
print(subclass_counts)


# --- Build layers with colors ---
for subclass_name, group in subclass_df.groupby("subclass", sort=True):
    body_ids = group["bodyId"].dropna().astype(int).tolist()
    if not body_ids:
        continue

    layer_name = f"MANC_v1.2.3_{subclass_name}"
    color = next(color_cycle)

    add_segmentation_layer(new_state, layer_name, body_ids, seg_source, color)


#add more manual layers based on my cluster review. 
#use a dictionary and loop to add the layers:
##these came from notebook 4
ir_2_04B_1_ids = []
ir_2_04B_2_ids = [] 
ir_2_04B_3_ids = [] 
ir_2_04B_4_ids =[] 
ir_2_04B_5_ids = []
ir_2_04B_6_ids = []

ir_1_1_ids = []
ir_1_2_ids = []
ir_1_3_ids = []
ir_1_4_ids = []
ir_1_5_ids = []
ir_1_6_ids = []

manual_layers = {

    "ir_2_04B_1": ir_2_04B_1_ids,
    "ir_2_04B_2": ir_2_04B_2_ids,
    "ir_2_04B_3": ir_2_04B_3_ids,
    "ir_2_04B_4": ir_2_04B_4_ids,
    "ir_2_04B_5": ir_2_04B_5_ids,
    "ir_2_04B_6": ir_2_04B_6_ids,
    "ir_1_1": ir_1_1_ids,
    "ir_1_2": ir_1_2_ids,
    "ir_1_3": ir_1_3_ids,
    "ir_1_4": ir_1_4_ids,
    "ir_1_5": ir_1_5_ids,
    "ir_1_6":ir_1_6_ids,

   
}

for name, ids in manual_layers.items():
    layer_name = f"MANC_v1.2.3_{name}"
    color = next(color_cycle)  # or set manually if you want control

    add_segmentation_layer(new_state, layer_name, ids, seg_source, color)



#Add asingle layer manually
# add_segmentation_layer(
#     new_state,
#     "MANC_v1.2.3_candidate_pair",
#     [123456789, 987654321],
#     seg_source,
#     "#FF0000"
# )



# --- Debug print ---
print("State title:", new_state["title"])
print("Number of layers:", len(new_state["layers"]))

for layer in new_state["layers"]:
    print(layer["name"], len(layer.get("segments", [])))


# --- Save ---
encoded = urllib.parse.quote(json.dumps(new_state, separators=(",", ":")))
NEW_URL = "https://clio-ng.janelia.org/#!" + encoded


STATE_OUT = OUTPUT_DIR / "MANC_v1.2.3_4B_04BT1r.json"
URL_OUT = OUTPUT_DIR / "MANC_v1.2.3_4B_04BT1r_url.txt"

with open(STATE_OUT, "w", encoding="utf-8") as f:
    json.dump(new_state, f, indent=2)

with open(URL_OUT, "w", encoding="utf-8") as f:
    f.write(NEW_URL)

print("\nSaved new state to:", STATE_OUT.resolve())
print("Saved URL to:", URL_OUT.resolve())
print("\nNew URL:\n")
print(NEW_URL)

EM source: {'url': 'precomputed://gs://flyem-vnc-2-26-213dba213ef26e094c16c860ae7f4be0/v3_emdata_clahe_xy/jpeg', 'subsources': {'default': True}, 'enableDefaultSubsources': False}
Seg source: [{'url': 'precomputed://gs://manc-seg-v1p2/manc-seg-v1.2', 'subsources': {'default': True, 'mesh': True}, 'enableDefaultSubsources': False}, {'url': 'precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/', 'subsources': {'default': True}, 'enableDefaultSubsources': False}, {'url': 'precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/', 'subsources': {'default': True}, 'enableDefaultSubsources': False}, {'url': 'precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/numeric_properties/', 'enableDefaultSubsources': False}]
subclass
IR    57
BR    28
II     4
BI     3
IA     3
Name: bodyId, dtype: int64
State title: MANC_v1.2.3_4B_T2r
Number of layers: 18
EM 0
MANC_v1.2.3_BI 3
MANC_v1.2.3_BR 28
MANC_v1.2.3_IA 3
MANC_v1.2.3_II 4
MANC_v1.2.3_

In [26]:
# split out the T2r neurons:

subclass_df = df_4b_t2_rhs.copy()

#retrieve bodyIDs per subclass:
subclass_df["subclass"] = (
    subclass_df["subclass"]
    .fillna("unclassified")
    .astype(str)
    .str.strip()
)
subclass_df.loc[subclass_df["subclass"] == "", "subclass"] = "unclassified"

subclass_to_ids = (
    subclass_df.groupby("subclass")["bodyId"]
    .apply(lambda s: sorted(set(s.dropna().astype(int))))
    .to_dict()
)

##inspect them
for subclass, ids in sorted(subclass_to_ids.items()):
    print(f"{subclass}: {len(ids)} cells")
    print(ids)
    print()



# build layers
for subclass_name, body_ids in sorted(subclass_to_ids.items()):
    layer_name = f"MANC_v1.2.3_{subclass_name}"
    color = next(color_cycle)
    add_segmentation_layer(new_state, layer_name, body_ids, seg_source, color)




BI: 3 cells
[15445, 16530, 19473]

BR: 28 cells
[11945, 14134, 14272, 14628, 14932, 15036, 15037, 15104, 15128, 16171, 16361, 16408, 16640, 16772, 16820, 16852, 16982, 17166, 17559, 17978, 18208, 18564, 18670, 20181, 20349, 23803, 32712, 101891]

IA: 3 cells
[10179, 10394, 15288]

II: 4 cells
[10873, 12284, 13720, 17981]

IR: 57 cells
[13591, 13608, 13668, 14256, 14423, 15109, 15537, 16396, 16459, 17386, 17435, 17956, 18335, 18574, 18663, 18788, 19071, 19484, 19497, 19641, 19844, 19901, 20046, 20152, 20187, 20346, 20416, 20556, 20650, 21120, 21146, 21147, 21216, 21272, 21459, 21464, 22132, 22299, 22342, 22780, 22793, 22800, 22988, 23309, 23358, 23973, 24942, 24986, 25864, 27855, 29101, 35471, 100075, 101038, 155096, 165061, 174628]



In [27]:
# select subclasses to include in final export

br_ids = subclass_to_ids.get("BR", [])
bi_ids = subclass_to_ids.get("BI", [])
ia_ids = subclass_to_ids.get("IA", [])
ii_ids = subclass_to_ids.get("II", [])

print("BR:", len(br_ids))
print("BI:", len(bi_ids))
print("IA:", len(ia_ids))
print("II:", len(ii_ids))

BR: 28
BI: 3
IA: 3
II: 4


In [8]:
#lets look at IR subclass liek I did for T1- is there dags inthe MANC description? 
#let me inspect subclass IR in 04B to see if MANC has subdivided it 

#fidn the annotations the way I see them when I select a neuron inthe API

from neuprint import fetch_neurons, NeuronCriteria as NC

criteria = NC(bodyId=df_4b_t2_rhs["bodyId"].tolist())

neurons, roi_counts = fetch_neurons(criteria, client=c)

print(neurons.columns)


#------------------------------------
## make neurons_clean should  to handle missing serialMotif values before grouping.

neurons_clean = neurons.copy()

# Flag whether a serial motif was originally annotated
neurons_clean["has_serialMotif"] = neurons_clean["serialMotif"].notna()

# Fill missing values so groupby does not drop them or leave NaN labels
neurons_clean["serialMotif"] = (
    neurons_clean["serialMotif"]
    .fillna("none")
    .astype(str)
    .str.strip()
)

# Optional cleanup for other grouping columns
# for col, default in {
#     "subclass": "unclassified",
#     "type": "unknown",
#     "instance": "none",
# }.items():
#     neurons_clean[col] = neurons_clean[col].fillna(default).astype(str).str.strip()
# table_with_ids = (
#     neurons_clean
#     .groupby(["subclass", "type", "instance", "serialMotif"])
#     .agg(
#         count=("bodyId", "nunique"),
#         bodyIds=("bodyId", lambda x: sorted(set(x)))
#     )
#     .reset_index()
# )
table = (
    neurons_clean
    .groupby(["subclass", "type", "instance", "serialMotif"])["bodyId"]
    .nunique()
    .reset_index(name="count")
)

# add flag AFTER grouping
table["has_motif"] = table["serialMotif"] != "none"

# sort
table = table.sort_values(
    by=["has_motif", "subclass", "count"],
    ascending=[False, True, False]
)

print(table)

#---------------------------------------

ir_df = neurons[neurons["subclass"] == "IR"].copy()
##temporary filtered view

ir_df_selectedcols = ir_df[[
    "bodyId",
    "type",
    "instance",
    "serialMotif",
    "description",
    "modality"
]].sort_values("type")

ir_df_selectedcols

Index(['bodyId', 'instance', 'type', 'pre', 'post', 'downstream', 'upstream',
       'size', 'status', 'statusLabel', 'somaLocation', 'roiInfo',
       'somaNeuromere', 'hemilineage', 'predictedNt', 'ntAcetylcholineProb',
       'class', 'vfbId', 'receptorType', 'synweight', 'ntGabaProb',
       'predictedNtProb', 'location', 'subcluster', 'birthtime', 'cluster',
       'celltypeTotalNtPredictions', 'subclassabbr', 'group', 'somaSide',
       'description', 'exitNerve', 'longTract', 'rootLocation', 'target',
       'systematicType', 'avgLocation', 'subclass', 'locationType', 'origin',
       'celltypePredictedNt', 'serialMotif', 'tosomaLocation', 'prefix',
       'serial', 'tag', 'entryNerve', 'modality', 'source', 'synonyms',
       'ntUnknownProb', 'transmission', 'rootSide', 'totalNtPredictions',
       'ntGlutamateProb', 'inputRois', 'outputRois'],
      dtype='object')
   subclass      type       instance      serialMotif  count  has_motif
2        BR  IN04B008  IN04B008_T2_R  ind

,bodyId,type,instance,serialMotif,description,modality
6,13608,IN04B011,IN04B011_T2_R,None,4B in T2 RHS,None
7,13668,IN04B011,IN04B011_T2_R,None,4B? in T2 RHS,None
93,165061,IN04B025,IN04B025_T2_R,independent leg,4B in T2 RHS,None
58,20187,IN04B026,IN04B026_T2_R,independent leg,4B in T2 RHS,None
5,13591,IN04B027,IN04B027_T2_R,None,None,None
86,29101,IN04B027,IN04B027_T2_R,None,None,None
22,15537,IN04B027,IN04B027_T2_R,None,None,None
36,17435,IN04B031,IN04B031_T2_R,independent leg,None,None
51,19497,IN04B037,IN04B037_T2_R,independent leg,4B? in T2 RHS,None
27,16459,IN04B049,IN04B049_T2_R,None,None,None


In [9]:
#ok let us make two Dfs for the 04bIR and the 20A flagged 04B IRs in T3L

#let me inspect subclass IR in 04B to see if MANC has subdivided it 

###pull all IR neurons in ir_df
ir_df = neurons[neurons["subclass"] == "IR"].copy()


ir_df[[
    "bodyId",
    "type",
    "instance",
    "serialMotif",
    "description",
    "modality"
]].sort_values("type")

ir_ids = sorted(ir_df["bodyId"].dropna().astype(int).unique().tolist())
print(ir_ids)
print("n IR neurons:", len(ir_ids))


# create flag
ir_df["maybe_20A"] = ir_df["description"].str.contains(
    "20A", case=False, na=False
)

# subset
ir_20A = ir_df[ir_df["maybe_20A"]]

ir_20A[[
    "bodyId", "type", "instance", "description", 'serialMotif'
]]

# everything else
ir_04B = ir_df[~ir_df["maybe_20A"]]



# make unique type lists for the two IR groups

types_20A = (
    ir_20A
    .groupby("type")["bodyId"]
    .agg(lambda x: sorted(set(x)))
    .reset_index(name="bodyIds")
    .sort_values("type")
    .reset_index(drop=True)
)

types_04B_or_unknown = (
    ir_04B
    .groupby("type")["bodyId"]
    .agg(lambda x: sorted(set(x)))
    .reset_index(name="bodyIds")
    .sort_values("type")
    .reset_index(drop=True)
)

types_20A
types_04B_or_unknown

types_20A_set = set(types_20A["type"])
types_04B_set = set(types_04B_or_unknown["type"])

overlap = types_20A_set & types_04B_set
only_20A = types_20A_set - types_04B_set
only_04B = types_04B_set - types_20A_set

print("Overlap:", overlap)
print("\nOnly in 20A:", only_20A)
print("\nOnly in 04B/unknown:", only_04B)


n_20A = ir_df[ir_df["maybe_20A"]]["bodyId"].nunique()
n_04B = ir_df[~ir_df["maybe_20A"]]["bodyId"].nunique()

print("IR maybe 20A:", n_20A)
print("IR 04B/unknown:", n_04B)

ids_20A = [int(x) for x in ir_20A["bodyId"].unique()]
ids_04B = [int(x) for x in ir_04B["bodyId"].unique()]

print(ids_20A)
print(ids_04B)



[13591, 13608, 13668, 14256, 14423, 15109, 15537, 16396, 16459, 17386, 17435, 17956, 18335, 18574, 18663, 18788, 19071, 19484, 19497, 19641, 19844, 19901, 20046, 20152, 20187, 20346, 20416, 20556, 20650, 21120, 21146, 21147, 21216, 21272, 21459, 21464, 22132, 22299, 22342, 22780, 22793, 22800, 22988, 23309, 23358, 23973, 24942, 24986, 25864, 27855, 29101, 35471, 100075, 101038, 155096, 165061, 174628]
n IR neurons: 57
Overlap: set()

Only in 20A: {'IN04B102', 'IN04B097', 'IN04B092', 'IN04B104', 'IN04B074', 'IN04B106', 'IN04B089', 'IN04B081', 'IN04B099', 'IN04B061', 'IN04B109', 'IN04B103', 'IN04B112', 'IN04B108'}

Only in 04B/unknown: {'IN04B082', 'IN04B087', 'IN04B031', 'IN04B071', 'IN04B062', 'IN04B027', 'IN04B049', 'IN04B090', 'IN04B100', 'IN04B078', 'IN04B025', 'IN04B026', 'IN04B084', 'IN04B011', 'IN04B037'}
IR maybe 20A: 26
IR 04B/unknown: 31
[14423, 15109, 16396, 18335, 18663, 19071, 19901, 20556, 20650, 21120, 21272, 21459, 22132, 22342, 22780, 22793, 22800, 23309, 23358, 23973, 

In [10]:
#Jelly manual cluster annotation- reviewed in neuroglancer for morphologies
ir_all = [13591, 13608, 13668, 14256, 14423, 15109, 15537, 16396, 16459, 17386, 17435, 17956, 18335, 18574, 18663, 18788, 19071, 19484, 19497, 19641, 19844, 19901, 20046, 20152, 20187, 20346, 20416, 20556, 20650, 21120, 21146, 21147, 21216, 21272, 21459, 21464, 22132, 22299, 22342, 22780, 22793, 22800, 22988, 23309, 23358, 23973, 24942, 24986, 25864, 27855, 29101, 35471, 100075, 101038, 155096, 165061, 174628]


print (len(ir_all))
ir_20A = [14423, 15109, 16396, 18335, 18663, 19071, 19901, 20556, 20650, 21120, 21272, 21459, 22132, 22342, 22780, 22793, 22800, 23309, 23358, 23973, 24942, 24986, 25864, 27855, 101038, 155096]


print (len(ir_20A))
ir_04B = [13591, 13608, 13668, 14256, 15537, 16459, 17386, 17435, 17956, 18574, 18788, 19484, 19497, 19641, 19844, 20046, 20152, 20187, 20346, 20416, 21146, 21147, 21216, 21464, 22299, 22988, 29101, 35471, 100075, 165061, 174628]


print (len(ir_04B))

#Jelly clusters: 
ir_2_04B_1_ids = [13591, 13608, 13668 ]
ir_2_04B_2_ids = [14256, 16459, 17956, 19484, 20152, 100075]
ir_2_04B_3_ids = [20187, 20416, 21216, 165061, 174628]
ir_2_04B_4_ids = [18574, 20046, 21464]
ir_2_04B_5_ids = [18788, 22299]
ir_2_04B_6_ids = [15537, 17435, 19497, 19641, 19844, 21146, 21147,29101, 35471]
ir_2_04B_7_ids = [17386,22988,20346] #20346 MISSES SOMA
ir_remaining = [x for x in ir_04B if x not in ir_2_04B_1_ids and x not in ir_2_04B_2_ids and x not in ir_2_04B_3_ids and x not in ir_2_04B_4_ids and x not in ir_2_04B_5_ids and x not in ir_2_04B_6_ids and x not in ir_2_04B_7_ids]
#ir_remaining = [x for x in ir_04B if x not in ir_2_04B_1_ids and x not in ir_2_04B_2_ids and x not in ir_2_04B_4_ids and x not in ir_2_04B_5_ids]

print(ir_remaining)



# # # #did I miss anything: 
all_groups = (
    set(ir_2_04B_1_ids)
    | set(ir_2_04B_2_ids)
    | set(ir_2_04B_3_ids)
    | set(ir_2_04B_4_ids)
    | set(ir_2_04B_5_ids)
    | set(ir_2_04B_6_ids)
    | set(ir_2_04B_7_ids)
)


ir_set = set(ir_04B)

missing = ir_set - all_groups
extra = all_groups - ir_set

print("Missing from groups:", missing)
print("Extra in groups:", extra)


#did I duplicate any neuron?
from collections import Counter

all_ids_list = (
    ir_2_04B_1_ids
    + ir_2_04B_2_ids
    + ir_2_04B_3_ids
    + ir_2_04B_4_ids
    + ir_2_04B_5_ids
    + ir_2_04B_6_ids
    + ir_2_04B_7_ids
)

counts = Counter(all_ids_list)

duplicates = {k: v for k, v in counts.items() if v > 1}
print("Duplicates across groups:", duplicates)



57
26
31
[]
Missing from groups: set()
Extra in groups: set()
Duplicates across groups: {}


In [8]:
#beautiful. Now we do group T1l-IR tagged as 20A

ir_all = [13591, 13608, 13668, 14256, 14423, 15109, 15537, 16396, 16459, 17386, 17435, 17956, 18335, 18574, 18663, 18788, 19071, 19484, 19497, 19641, 19844, 19901, 20046, 20152, 20187, 20346, 20416, 20556, 20650, 21120, 21146, 21147, 21216, 21272, 21459, 21464, 22132, 22299, 22342, 22780, 22793, 22800, 22988, 23309, 23358, 23973, 24942, 24986, 25864, 27855, 29101, 35471, 100075, 101038, 155096, 165061, 174628]


print (len(ir_all))
ir_20A = [14423, 15109, 16396, 18335, 18663, 19071, 19901, 20556, 20650, 21120, 21272, 21459, 22132, 22342, 22780, 22793, 22800, 23309, 23358, 23973, 24942, 24986, 25864, 27855, 101038, 155096]


print (len(ir_20A))
ir_04B = [13591, 13608, 13668, 14256, 15537, 16459, 17386, 17435, 17956, 18574, 18788, 19484, 19497, 19641, 19844, 20046, 20152, 20187, 20346, 20416, 21146, 21147, 21216, 21464, 22299, 22988, 29101, 35471, 100075, 165061, 174628]


print (len(ir_04B))



ir_1_1_ids = [13591, 13608, 19497, 19641, 19844, 21146]

ir_1_2_ids = [14423, 15109]
ir_1_3_ids = [16396, 21120, 21459, 22132, 22342, 22780, 23973, 24986, 27855, 155096]

ir_1_4_ids = [18335, 18663, 20650]
ir_1_5_ids = [19071, 20556, 21272, 22793, 23358, 24942, 25864]
ir_1_6_ids = [23309, 101038]
ir_1_7_ids = [19901, 22800]
ir_1_remaining = [x for x in ir_20A if x not in ir_1_1_ids and x not in ir_1_2_ids  and x not in ir_1_3_ids and x not in ir_1_4_ids and x not in ir_1_5_ids and x not in ir_1_6_ids and x not in ir_1_7_ids]

#ir_1_remaining = [x for x in ir_20A if x not in ir_1_1_ids and x not in ir_1_2_ids and x not in ir_1_3_ids and x not in ir_1_4_ids and x not in ir_1_5_ids] 
print("IDs IR remaining:", ir_1_remaining)

# #did I miss anything: 
all_groups = (
    set(ir_1_1_ids)
    | set(ir_1_2_ids)
    | set(ir_1_3_ids)
    | set(ir_1_4_ids)
    | set(ir_1_5_ids)
    | set(ir_1_6_ids)
    | set(ir_1_7_ids)
)


ir_set_20A = set(ir_20A)

missing = ir_set_20A - all_groups
extra = all_groups - ir_set_20A

print("Missing from groups:", missing)
print("Extra in groups:", extra)


#did I duplicate any neuron?
from collections import Counter

all_ids_list = (
    ir_1_1_ids
    + ir_1_2_ids
    + ir_1_3_ids
    + ir_1_4_ids
    + ir_1_5_ids
    + ir_1_6_ids
    + ir_1_7_ids
  
)

counts = Counter(all_ids_list)

duplicates = {k: v for k, v in counts.items() if v > 1}
print("Duplicates across groups:", duplicates)




57
26
31
IDs IR remaining: []
Missing from groups: set()
Extra in groups: {19844, 13608, 19497, 13591, 19641, 21146}
Duplicates across groups: {}


In [32]:
#build dataframe for export with biological information

neuromere = "T2R"

meta_df = df_4b_t2_rhs.copy()
meta_df.head()


#define manual assignments
manual_assignments = {
    #IR
    "ir_1_1_ids": ir_1_1_ids,
    "ir_1_2_ids": ir_1_2_ids,
    "ir_1_3_ids": ir_1_3_ids,
    "ir_1_4_ids": ir_1_4_ids,
    "ir_1_5_ids": ir_1_5_ids,
    "ir_1_6_ids": ir_1_6_ids,
    "ir_1_7_ids": ir_1_7_ids,
    "ir_2_04B_1_ids": ir_2_04B_1_ids,
    "ir_2_04B_2_ids": ir_2_04B_2_ids,
    "ir_2_04B_3_ids": ir_2_04B_3_ids,
    "ir_2_04B_4_ids": ir_2_04B_4_ids,
    "ir_2_04B_5_ids": ir_2_04B_5_ids,
    "ir_2_04B_6_ids": ir_2_04B_6_ids,
    "ir_2_04B_7_ids": ir_2_04B_7_ids,

    #other subclass groups
    "br_ids": br_ids,
    "bi_ids": bi_ids,
    "ia_ids": ia_ids,
    "ii_ids": ii_ids,

}



# define how to interpret each group name

import re

def classify_group_var(group_var):
    # IR branch tagged as 20A
    m1 = re.match(r"ir_1_(\d+)_ids$", group_var)
    if m1:
        return "IR", "20A", int(m1.group(1))

    # IR branch tagged as 04B
    m2 = re.match(r"ir_2_04B_(\d+)_ids$", group_var)
    if m2:
        return "IR", "04B", int(m2.group(1))

    # broad subclass groups
    if group_var == "ba_ids":
        return "Ba", None, None

    if group_var == "br_ids":
        return "BR", None, None

    if group_var == "bi_ids":
        return "BI", None, None

    if group_var == "ia_ids":
        return "IA", None, None

    if group_var == "ii_ids":
        return "II", None, None

    return None, None, None




# make row-wise assignment dataframe
rows = []

for var_name, body_ids in manual_assignments.items():
    if not isinstance(body_ids, list):
        continue

    major_class, subgroup, morphology_cluster = classify_group_var(var_name)

    for body_id in body_ids:
        rows.append({
            "neuromere": neuromere,
            "bodyId": int(body_id),
            "group_var": var_name,
            "major_class": major_class,
            "subgroup": subgroup,
            "morphology_cluster": morphology_cluster,
        })

assignment_df = pd.DataFrame(rows)

print(assignment_df.shape)
assignment_df.head()
assignment_df["major_class"].unique()


(43, 6)


array(['IR', 'BR', 'BI', 'IA', 'II'], dtype=object)

In [ ]:
#merge metadata
final_ir_df = assignment_df.merge(
    meta_df[
        [
            "bodyId",
            "type",
            "instance",
            "class",
            "subclass",
            "hemilineage",
            "somaNeuromere",
            "somaSide",
            "rootSide",
            "longTract",
            "birthtime",
        ]
    ],
    on="bodyId",
    how="left"
)

print(final_ir_df.shape)
final_ir_df.head()

# a few checks:
final_ir_df.groupby(["legacy_tag", "morphology_cluster"])["bodyId"].nunique()

#check for duplicate assignments
dup_check = (
    final_ir_df.groupby(["bodyId"])["group_var"]
    .nunique()
    .reset_index(name="n_groups")
)

dup_check[dup_check["n_groups"] > 1]



(63, 16)


,bodyId,n_groups
0,13591,2
1,13608,2
18,19497,2
19,19641,2
20,19844,2
30,21146,2


#final sort 
https://clio-ng.janelia.org/#!%7B%22title%22:%22MANC_v1.2.3_4B_T2r%22%2C%22dimensions%22:%7B%22x%22:%5B8e-9%2C%22m%22%5D%2C%22y%22:%5B8e-9%2C%22m%22%5D%2C%22z%22:%5B8e-9%2C%22m%22%5D%7D%2C%22position%22:%5B29696%2C33792%2C33792%5D%2C%22crossSectionOrientation%22:%5B0%2C0.7071067690849304%2C-0.7071067690849304%2C0%5D%2C%22crossSectionScale%22:1%2C%22projectionOrientation%22:%5B0.2439929097890854%2C0.30736225843429565%2C-0.36971747875213623%2C-0.8422024250030518%5D%2C%22projectionScale%22:27518.321388470133%2C%22layers%22:%5B%7B%22type%22:%22image%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://flyem-vnc-2-26-213dba213ef26e094c16c860ae7f4be0/v3_emdata_clahe_xy/jpeg%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22tab%22:%22source%22%2C%22name%22:%22EM%22%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%5B%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22subsources%22:%7B%22default%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%22%2C%22enableDefaultSubsources%22:false%7D%5D%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%2215445%22%2C%2216530%22%2C%2219473%22%5D%2C%22segmentQuery%22:%2215445%2C%2016530%2C%2019473%22%2C%22segmentColors%22:%7B%2215445%22:%22#87ceeb%22%2C%2216530%22:%22#87ceeb%22%2C%2219473%22:%22#87ceeb%22%7D%2C%22name%22:%22MANC_v1.2.3_BI%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%5B%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22subsources%22:%7B%22default%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%22%2C%22enableDefaultSubsources%22:false%7D%5D%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%22101891%22%2C%2211945%22%2C%2214134%22%2C%2214272%22%2C%2214628%22%2C%2214932%22%2C%2215036%22%2C%2215037%22%2C%2215104%22%2C%2215128%22%2C%2216171%22%2C%2216361%22%2C%2216408%22%2C%2216640%22%2C%2216772%22%2C%2216820%22%2C%2216852%22%2C%2216982%22%2C%2217166%22%2C%2217559%22%2C%2217978%22%2C%2218208%22%2C%2218564%22%2C%2218670%22%2C%2220181%22%2C%2220349%22%2C%2223803%22%2C%2232712%22%5D%2C%22segmentQuery%22:%2211945%2C%2014134%2C%2014272%2C%2014628%2C%2014932%2C%2015036%2C%2015037%2C%2015104%2C%2015128%2C%2016171%2C%2016361%2C%2016408%2C%2016640%2C%2016772%2C%2016820%2C%2016852%2C%2016982%2C%2017166%2C%2017559%2C%2017978%2C%2018208%2C%2018564%2C%2018670%2C%2020181%2C%2020349%2C%2023803%2C%2032712%2C%20101891%22%2C%22segmentColors%22:%7B%2211945%22:%22#ffa500%22%2C%2214134%22:%22#ffa500%22%2C%2214272%22:%22#ffa500%22%2C%2214628%22:%22#ffa500%22%2C%2214932%22:%22#ffa500%22%2C%2215036%22:%22#ffa500%22%2C%2215037%22:%22#ffa500%22%2C%2215104%22:%22#ffa500%22%2C%2215128%22:%22#ffa500%22%2C%2216171%22:%22#ffa500%22%2C%2216361%22:%22#ffa500%22%2C%2216408%22:%22#ffa500%22%2C%2216640%22:%22#ffa500%22%2C%2216772%22:%22#ffa500%22%2C%2216820%22:%22#ffa500%22%2C%2216852%22:%22#ffa500%22%2C%2216982%22:%22#ffa500%22%2C%2217166%22:%22#ffa500%22%2C%2217559%22:%22#ffa500%22%2C%2217978%22:%22#ffa500%22%2C%2218208%22:%22#ffa500%22%2C%2218564%22:%22#ffa500%22%2C%2218670%22:%22#ffa500%22%2C%2220181%22:%22#ffa500%22%2C%2220349%22:%22#ffa500%22%2C%2223803%22:%22#ffa500%22%2C%2232712%22:%22#ffa500%22%2C%22101891%22:%22#ffa500%22%7D%2C%22name%22:%22MANC_v1.2.3_BR%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%5B%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22subsources%22:%7B%22default%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%22%2C%22enableDefaultSubsources%22:false%7D%5D%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%2210179%22%2C%2210394%22%2C%2215288%22%5D%2C%22segmentQuery%22:%2210179%2C%2010394%2C%2015288%22%2C%22segmentColors%22:%7B%2210179%22:%22#e34234%22%2C%2210394%22:%22#e34234%22%2C%2215288%22:%22#e34234%22%7D%2C%22name%22:%22MANC_v1.2.3_IA%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%5B%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22subsources%22:%7B%22default%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%22%2C%22enableDefaultSubsources%22:false%7D%5D%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%2210873%22%2C%2212284%22%2C%2213720%22%2C%2217981%22%5D%2C%22segmentQuery%22:%2210873%2C%2012284%2C%2013720%2C%2017981%22%2C%22segmentColors%22:%7B%2210873%22:%22#cc99ff%22%2C%2212284%22:%22#cc99ff%22%2C%2213720%22:%22#cc99ff%22%2C%2217981%22:%22#cc99ff%22%7D%2C%22name%22:%22MANC_v1.2.3_II%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%5B%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22subsources%22:%7B%22default%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%22%2C%22enableDefaultSubsources%22:false%7D%5D%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%22100075%22%2C%22101038%22%2C%2213591%22%2C%2213608%22%2C%2213668%22%2C%2214256%22%2C%2214423%22%2C%2215109%22%2C%22155096%22%2C%2215537%22%2C%2216396%22%2C%2216459%22%2C%22165061%22%2C%2217386%22%2C%2217435%22%2C%22174628%22%2C%2217956%22%2C%2218335%22%2C%2218574%22%2C%2218663%22%2C%2218788%22%2C%2219071%22%2C%2219484%22%2C%2219497%22%2C%2219641%22%2C%2219844%22%2C%2219901%22%2C%2220046%22%2C%2220152%22%2C%2220187%22%2C%2220346%22%2C%2220416%22%2C%2220556%22%2C%2220650%22%2C%2221120%22%2C%2221146%22%2C%2221147%22%2C%2221216%22%2C%2221272%22%2C%2221459%22%2C%2221464%22%2C%2222132%22%2C%2222299%22%2C%2222342%22%2C%2222780%22%2C%2222793%22%2C%2222800%22%2C%2222988%22%2C%2223309%22%2C%2223358%22%2C%2223973%22%2C%2224942%22%2C%2224986%22%2C%2225864%22%2C%2227855%22%2C%2229101%22%2C%2235471%22%5D%2C%22segmentQuery%22:%2213591%2C%2013608%2C%2013668%2C%2014256%2C%2014423%2C%2015109%2C%2015537%2C%2016396%2C%2016459%2C%2017386%2C%2017435%2C%2017956%2C%2018335%2C%2018574%2C%2018663%2C%2018788%2C%2019071%2C%2019484%2C%2019497%2C%2019641%2C%2019844%2C%2019901%2C%2020046%2C%2020152%2C%2020187%2C%2020346%2C%2020416%2C%2020556%2C%2020650%2C%2021120%2C%2021146%2C%2021147%2C%2021216%2C%2021272%2C%2021459%2C%2021464%2C%2022132%2C%2022299%2C%2022342%2C%2022780%2C%2022793%2C%2022800%2C%2022988%2C%2023309%2C%2023358%2C%2023973%2C%2024942%2C%2024986%2C%2025864%2C%2027855%2C%2029101%2C%2035471%2C%20100075%2C%20101038%2C%20155096%2C%20165061%2C%20174628%22%2C%22segmentColors%22:%7B%2213591%22:%22#66cdaa%22%2C%2213608%22:%22#66cdaa%22%2C%2213668%22:%22#66cdaa%22%2C%2214256%22:%22#66cdaa%22%2C%2214423%22:%22#66cdaa%22%2C%2215109%22:%22#66cdaa%22%2C%2215537%22:%22#66cdaa%22%2C%2216396%22:%22#66cdaa%22%2C%2216459%22:%22#66cdaa%22%2C%2217386%22:%22#66cdaa%22%2C%2217435%22:%22#66cdaa%22%2C%2217956%22:%22#66cdaa%22%2C%2218335%22:%22#66cdaa%22%2C%2218574%22:%22#66cdaa%22%2C%2218663%22:%22#66cdaa%22%2C%2218788%22:%22#66cdaa%22%2C%2219071%22:%22#66cdaa%22%2C%2219484%22:%22#66cdaa%22%2C%2219497%22:%22#66cdaa%22%2C%2219641%22:%22#66cdaa%22%2C%2219844%22:%22#66cdaa%22%2C%2219901%22:%22#66cdaa%22%2C%2220046%22:%22#66cdaa%22%2C%2220152%22:%22#66cdaa%22%2C%2220187%22:%22#66cdaa%22%2C%2220346%22:%22#66cdaa%22%2C%2220416%22:%22#66cdaa%22%2C%2220556%22:%22#66cdaa%22%2C%2220650%22:%22#66cdaa%22%2C%2221120%22:%22#66cdaa%22%2C%2221146%22:%22#66cdaa%22%2C%2221147%22:%22#66cdaa%22%2C%2221216%22:%22#66cdaa%22%2C%2221272%22:%22#66cdaa%22%2C%2221459%22:%22#66cdaa%22%2C%2221464%22:%22#66cdaa%22%2C%2222132%22:%22#66cdaa%22%2C%2222299%22:%22#66cdaa%22%2C%2222342%22:%22#66cdaa%22%2C%2222780%22:%22#66cdaa%22%2C%2222793%22:%22#66cdaa%22%2C%2222800%22:%22#66cdaa%22%2C%2222988%22:%22#66cdaa%22%2C%2223309%22:%22#66cdaa%22%2C%2223358%22:%22#66cdaa%22%2C%2223973%22:%22#66cdaa%22%2C%2224942%22:%22#66cdaa%22%2C%2224986%22:%22#66cdaa%22%2C%2225864%22:%22#66cdaa%22%2C%2227855%22:%22#66cdaa%22%2C%2229101%22:%22#66cdaa%22%2C%2235471%22:%22#66cdaa%22%2C%22100075%22:%22#66cdaa%22%2C%22101038%22:%22#66cdaa%22%2C%22155096%22:%22#66cdaa%22%2C%22165061%22:%22#66cdaa%22%2C%22174628%22:%22#66cdaa%22%7D%2C%22name%22:%22MANC_v1.2.3_IR%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%5B%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22subsources%22:%7B%22default%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%22%2C%22enableDefaultSubsources%22:false%7D%5D%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%2213591%22%2C%2213608%22%2C%2213668%22%5D%2C%22segmentQuery%22:%2213591%2C%2013608%2C%2013668%22%2C%22name%22:%22MANC_v1.2.3_ir_2_04B_1%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%5B%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22subsources%22:%7B%22default%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%22%2C%22enableDefaultSubsources%22:false%7D%5D%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%22100075%22%2C%2214256%22%2C%2216459%22%2C%2217956%22%2C%2219484%22%2C%2220152%22%5D%2C%22segmentQuery%22:%2214256%2C%2016459%2C%2017956%2C%2019484%2C%2020152%2C%20100075%22%2C%22name%22:%22MANC_v1.2.3_ir_2_04B_2%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%5B%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22subsources%22:%7B%22default%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%22%2C%22enableDefaultSubsources%22:false%7D%5D%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%22165061%22%2C%22174628%22%2C%2220187%22%2C%2220416%22%2C%2221216%22%5D%2C%22segmentQuery%22:%2220187%2C%2020416%2C%2021216%2C%20165061%2C%20174628%22%2C%22name%22:%22MANC_v1.2.3_ir_2_04B_3%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%5B%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22subsources%22:%7B%22default%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%22%2C%22enableDefaultSubsources%22:false%7D%5D%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%2218574%22%2C%2220046%22%2C%2221464%22%5D%2C%22segmentQuery%22:%2218574%2C%2020046%2C%2021464%22%2C%22name%22:%22MANC_v1.2.3_ir_2_04B_4%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%5B%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22subsources%22:%7B%22default%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%22%2C%22enableDefaultSubsources%22:false%7D%5D%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%2218788%22%2C%2222299%22%5D%2C%22segmentQuery%22:%2218788%2C%2022299%22%2C%22name%22:%22MANC_v1.2.3_ir_2_04B_5%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%5B%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22subsources%22:%7B%22default%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%22%2C%22enableDefaultSubsources%22:false%7D%5D%2C%22tab%22:%22source%22%2C%22segments%22:%5B%2215537%22%2C%2217435%22%2C%2219497%22%2C%2219641%22%2C%2219844%22%2C%2221146%22%2C%2221147%22%2C%2229101%22%2C%2235471%22%5D%2C%22segmentQuery%22:%2215537%2C%2017435%2C%2019497%2C%2019641%2C%2019844%2C%2021146%2C%2021147%2C%2029101%2C%2035471%22%2C%22name%22:%22MANC_v1.2.3_ir_2_04B_6%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%2217386%22%2C%2220346%22%2C%2222988%22%5D%2C%22segmentQuery%22:%2217386%2C22988%2C20346%22%2C%22name%22:%22MANC_v1.2.3_ir_2_04B_7%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%5B%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22subsources%22:%7B%22default%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%22%2C%22enableDefaultSubsources%22:false%7D%5D%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%2219497%22%2C%2219844%22%5D%2C%22segmentQuery%22:%2213591%2C%2013608%2C%2019497%2C%2019641%2C%2019844%2C%2021146%22%2C%22name%22:%22MANC_v1.2.3_ir_1_1%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%5B%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22subsources%22:%7B%22default%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%22%2C%22enableDefaultSubsources%22:false%7D%5D%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%2214423%22%5D%2C%22segmentQuery%22:%2214423%2C%2015109%22%2C%22name%22:%22MANC_v1.2.3_ir_1_2%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%5B%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22subsources%22:%7B%22default%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%22%2C%22enableDefaultSubsources%22:false%7D%5D%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%22155096%22%2C%2216396%22%2C%2221120%22%2C%2221459%22%2C%2222132%22%2C%2222342%22%2C%2222780%22%2C%2223973%22%2C%2224986%22%2C%2227855%22%5D%2C%22segmentQuery%22:%2216396%2C%2021120%2C%2021459%2C%2022132%2C%2022342%2C%2022780%2C%2023973%2C%2024986%2C%2027855%2C%20155096%22%2C%22name%22:%22MANC_v1.2.3_ir_1_3%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%5B%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22subsources%22:%7B%22default%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%22%2C%22enableDefaultSubsources%22:false%7D%5D%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%2218335%22%2C%2218663%22%2C%2220650%22%5D%2C%22segmentQuery%22:%2218335%2C%2018663%2C%2020650%22%2C%22name%22:%22MANC_v1.2.3_ir_1_4%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%5B%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22subsources%22:%7B%22default%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%22%2C%22enableDefaultSubsources%22:false%7D%5D%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%2219071%22%2C%2220556%22%2C%2221272%22%2C%2222793%22%2C%2223358%22%2C%2224942%22%2C%2225864%22%5D%2C%22segmentQuery%22:%2219071%2C%2020556%2C%2021272%2C%2022793%2C%2023358%2C%2024942%2C%2025864%22%2C%22name%22:%22MANC_v1.2.3_ir_1_5%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%5B%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22subsources%22:%7B%22default%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%22%2C%22enableDefaultSubsources%22:false%7D%5D%2C%22tab%22:%22source%22%2C%22segments%22:%5B%22101038%22%2C%2223309%22%5D%2C%22segmentQuery%22:%2223309%2C%20101038%22%2C%22name%22:%22MANC_v1.2.3_ir_1_6%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%2219901%22%2C%2222800%22%5D%2C%22segmentQuery%22:%2219901%2C%2022800%22%2C%22name%22:%22MANC_v1.2.3_ir_1_7%22%7D%5D%2C%22showSlices%22:false%2C%22selectedLayer%22:%7B%22flex%22:1.49%2C%22visible%22:true%2C%22layer%22:%22MANC_v1.2.3_ir_1_7%22%7D%2C%22crossSectionBackgroundColor%22:%22#ffffff%22%2C%22projectionBackgroundColor%22:%22#ffffff%22%2C%22layout%22:%223d%22%2C%22layerListPanel%22:%7B%22visible%22:true%7D%7D